# MASLD-Bench · 02 — Frontier Evaluation on Colab (Drive-synced edition)

The frontier bracket is already running locally (DeepSeek + Gemini complete). This Colab version is for **re-runs and extensions** — e.g. the models that 402'd locally — reading prompts from your Drive-synced folders and writing results back.

Protocol: frozen ANALYSIS_PLAN_v1 — zero-shot, greedy, single-pass, 300/task.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import os# Drive for Desktop computers-sync mounts at /content/drive/Othercomputers/My Mac/# (read-only from Colab); mirrored copies may also exist under MyDrive.SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']def find_dir(target_name, search_roots=None, depth=6):    if search_roots is None:        search_roots = SEARCH_ROOTS    matches = []    # fast path: direct child of a sync root    for root in search_roots:        direct = os.path.join(root, target_name)        if os.path.isdir(direct):            matches.append(direct)    if matches:        return matches    def walk(path, d):        if d > depth: return        try:            for entry in os.scandir(path):                if entry.is_dir(follow_symlinks=False):                    if entry.name == target_name:                        matches.append(entry.path)                    walk(entry.path, d + 1)        except (PermissionError, OSError):            pass    for root in search_roots:        if os.path.exists(root):            walk(root, 0)    return matchescands = find_dir('masld-llm')MASLD_LLM = min(cands, key=len) if cands else '/content/drive/MyDrive/masld-llm'print('MASLD_LLM =', MASLD_LLM)def is_writable(path):    try:        probe = os.path.join(path, '.write_test')        open(probe, 'w').close(); os.remove(probe); return True    except Exception: return FalseRUNS = os.path.join(MASLD_LLM, 'results', 'frontier_runs') if is_writable(os.path.join(MASLD_LLM, 'results')) else '/content/drive/MyDrive/masld-llm-outputs/frontier_runs'os.makedirs(RUNS, exist_ok=True)print('RUNS =', RUNS)print('prompts present:', os.path.exists(os.path.join(MASLD_LLM, 'prompts', 'MB-20xx_liver_test.jsonl.gz')))

In [ ]:
# copy the local runner + scorer into Colab (they're synced via Drive)!cp "$MASLD_LLM/scripts/session3_frontier.py" /content/session3_frontier.py!cp "$MASLD_LLM/scripts/score_frontier.py" /content/score_frontier.pyimport os, secrets# OpenRouter key - paste when prompted, or rely on Colab secrets (key icon)os.environ['OPENROUTER_API_KEY'] = input('OpenRouter key: ').strip() or os.environ.get('OPENROUTER_API_KEY', '')

In [ ]:
# run a model (resumable; replaces the local 402-failed runs)%cd /content!python3 session3_frontier.py openai/gpt-4.1-mini 300# then: google/gemini-2.5-pro (if extending), anthropic/claude-sonnet-4.5, x-ai/grok-4-fast# cheap backups if 402 persists: moonshotai/kimi-k3, minimax/minimax-m3

In [ ]:
# score everything -> leaderboard!python3 score_frontier.pyimport pandas as pdtry:    df = pd.read_csv(f"{MASLD_LLM}/results/leaderboard.csv")    print(df.to_string(index=False))except Exception as e:    print('leaderboard not written yet:', e)